In [1]:
import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import *

# call de API
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2026-01-01",
    "end_date": "2026-02-28",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data = response.json()

# print data
print(data.keys())
print(data["daily"].keys())

StatementMeta(, cd3b313d-7352-40ba-a001-66e7c8036953, 3, Finished, Available, Finished, False)

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'snowfall_sum', 'wind_speed_10m_max'])


In [2]:
import pandas as pd

#convert in DF
pdf = pd.DataFrame(data["daily"])
df_weather = spark.createDataFrame(pdf)
df_weather.show(5)
df_weather.printSchema()

StatementMeta(, cd3b313d-7352-40ba-a001-66e7c8036953, 4, Finished, Available, Finished, False)

+----------+------------------+------------------+-----------------+------------+------------------+
|      time|temperature_2m_max|temperature_2m_min|precipitation_sum|snowfall_sum|wind_speed_10m_max|
+----------+------------------+------------------+-----------------+------------+------------------+
|2026-01-01|               0.6|              -7.1|              1.1|        0.77|              25.1|
|2026-01-02|              -1.2|              -7.3|              0.3|        0.21|              14.4|
|2026-01-03|              -0.6|              -6.7|              0.0|         0.0|              10.3|
|2026-01-04|               1.0|              -4.4|              0.3|        0.21|              12.1|
|2026-01-05|               0.9|              -5.4|              0.8|        0.56|              12.6|
+----------+------------------+------------------+-----------------+------------+------------------+
only showing top 5 rows

root
 |-- time: string (nullable = true)
 |-- temperature_2m_max: 

In [3]:
#save table
df_weather.write.format("delta").mode("overwrite").saveAsTable("weather_bronze")

StatementMeta(, cd3b313d-7352-40ba-a001-66e7c8036953, 5, Finished, Available, Finished, False)